In [ ]:
!pip install pandas numpy plotly -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
cause_score = {

    "vehicle_breakdown":2,

    "pot_holes":2,

    "construction":3,

    "water_logging":3,

    "road_conditions":3,

    "tree_fall":3,

    "accident":4,

    "congestion":4,

    "public_event":5,

    "procession":5,

    "vip_movement":5,

    "protest":5,

    "others":2,

    "debris":2
}

In [ ]:
risk_multiplier = {

    "Low":1,

    "Medium":1.5,

    "High":2,

    "Critical":3
}

In [ ]:
import numpy as np

def simulate_event(

    event_cause,
    priority,
    road_closure,
    risk_level,
    duration_hours,
    crowd_size

):

    # Event severity

    cause_score = {

        "vehicle_breakdown":1,

        "pot_holes":1,

        "construction":2,

        "water_logging":2,

        "road_conditions":2,

        "tree_fall":2,

        "accident":3,

        "congestion":3,

        "public_event":4,

        "procession":4,

        "vip_movement":5,

        "protest":5,

        "others":1,

        "debris":1
    }

    base = cause_score.get(
        event_cause.lower(),
        2
    )

    # Priority impact

    if priority.lower() == "high":
        base += 2

    # Road closure impact

    if road_closure:
        base += 2

    # Risk multiplier from Module 1

    risk_multiplier = {

        "Low":1.0,

        "Medium":1.3,

        "High":1.6,

        "Critical":2.0
    }

    base = (
        base *
        risk_multiplier[risk_level]
    )

    # Crowd scaling

    crowd_factor = (
        np.log1p(crowd_size)
        /
        np.log1p(1000)
    )

    # Duration scaling

    duration_factor = (
        1 +
        duration_hours / 12
    )

    # Final impact score

    impact_score = (
        base *
        crowd_factor *
        duration_factor
    )

    # Resource recommendation

    officers_mid = min(
        impact_score * 1.5,
        40
    )

    barricades_mid = min(
        impact_score * 2.5,
        70
    )

    # Range instead of fixed value

    officers_min = round(
        officers_mid * 0.8
    )

    officers_max = round(
        officers_mid * 1.2
    )

    barricades_min = round(
        barricades_mid * 0.8
    )

    barricades_max = round(
        barricades_mid * 1.2
    )

    # Response classification

    if impact_score < 6:

        response = "Routine"

    elif impact_score < 12:

        response = "Urgent"

    else:

        response = "Critical"

    return {

        "response":response,

        "impact_score":
        round(
            impact_score,
            2
        ),

        "officers_range":
        f"{officers_min}-{officers_max}",

        "barricades_range":
        f"{barricades_min}-{barricades_max}"
    }

In [ ]:
simulate_event(

    event_cause="procession",

    priority="High",

    road_closure=True,

    risk_level="Critical",

    duration_hours=4,

    crowd_size=5000
)

{'response': 'Critical',
 'impact_score': np.float64(26.3),
 'officers_range': '32-47',
 'barricades_range': '53-79'}

In [ ]:
simulate_event(
    "vehicle_breakdown",
    "Low",
    False,
    "Low",
    1,
    100
)

{'response': 'Routine',
 'impact_score': np.float64(0.72),
 'officers_range': '1-1',
 'barricades_range': '1-2'}

In [ ]:
durations = [
    1,
    2,
    4,
    6,
    8
]

results = []

for d in durations:

    r = simulate_event(

        event_cause="procession",

        priority="High",

        road_closure=True,

        risk_level="Critical",

        duration_hours=d,

        crowd_size=5000
    )

    results.append({

        "duration":d,

        "officers":
        r["officers"],

        "barricades":
        r["barricades"]
    })

In [ ]:
sim_df = pd.DataFrame(
    results
)

sim_df

,duration,officers,barricades
0,1,101,135
1,2,202,270
2,4,405,540
3,6,608,810
4,8,810,1080


In [ ]:
fig = px.line(

    sim_df,

    x="duration",

    y="officers",

    markers=True,

    title=
    "Officers Needed vs Event Duration"
)

fig.show()

In [ ]:
fig = px.line(

    sim_df,

    x="duration",

    y="barricades",

    markers=True,

    title=
    "Barricades Needed vs Event Duration"
)

fig.show()

In [ ]:
crowds = [

    500,

    1000,

    3000,

    5000,

    10000
]

crowd_results = []

for c in crowds:

    r = simulate_event(

        event_cause="public_event",

        priority="High",

        road_closure=True,

        risk_level="Critical",

        duration_hours=4,

        crowd_size=c
    )

    crowd_results.append({

        "crowd_size":c,

        "officers":
        r["officers"],

        "barricades":
        r["barricades"]
    })

In [ ]:
crowd_df = pd.DataFrame(
    crowd_results
)

crowd_df

,crowd_size,officers,barricades
0,500,40,54
1,1000,81,108
2,3000,243,324
3,5000,405,540
4,10000,810,1080


In [ ]:
fig = px.bar(

    crowd_df,

    x="crowd_size",

    y="officers",

    title=
    "Officers Required vs Crowd Size"
)

fig.show()

In [ ]:
import joblib

joblib.dump(
    simulate_event,
    "what_if_simulator.pkl"
)

['what_if_simulator.pkl']